## 01 — Build a custom AML environment (container) from Dockerfile

Builds an Azure ML custom environment from `container/Dockerfile` and
`container/requirements.txt` using the AML Python SDK, then triggers the
actual ACR build by submitting a smoke-test job on the AML compute cluster.

The resulting environment can be referenced by a training job (e.g.
`aml_cc_finetune_llama3.ipynb`) as `environment="<env_name>@latest"`.

### Prerequisite

**Run `00_aml_prepare_sub_environment.ipynb` first** (or confirm it has been
run for this machine + IP). That notebook sets up:
- NSP rule for your IP (so blob uploads work)
- NSP rule for the AML subscription (so ACR Tasks can read the build context)
- `allowSharedKeyAccess=true` + `publicNetworkAccess=Enabled` on storage
- Policy exemption so the MCAPS Gov policy doesn't revert shared-key

If you skip the prep notebook, the env upload will time out or jobs will
fail with `Key based authentication is not permitted`.

### Workflow

`BuildContext` works in two steps:

1. **Upload** the local build context folder (`./container`) to the
   workspace's default blob store — over HTTPS, authenticated as you with
   Entra ID.
2. **Trigger an ACR build** on the workspace's container registry, which
   pulls the uploaded folder, runs `docker build`, and publishes the image
   as a new environment version.

AML defers the actual image build until the environment is **first used by a
job**. The smoke-test cell at the bottom submits a tiny `python -c "..."` job
on `Cluster-A100-1GPU` to force the build and verify the resulting container.

### Reference
* Environment (BuildContext, Dockerfile): https://learn.microsoft.com/en-us/azure/machine-learning/how-to-manage-environments-v2?view=azureml-api-2&tabs=python
* Curated ACPT base images: https://onnxruntime.ai/docs/ecosystem/ptca_image_list.html


In [1]:
import azure.ai.ml as aml
print(f"Azure ML SDK version: {aml.__version__}")

Azure ML SDK version: 1.33.0


In [2]:
# get a handle to the workspace
import os
from azure.ai.ml import MLClient
from dotenv import load_dotenv

config_file_name = "germanywest.env"
config_file_path = os.path.join(".", "config", config_file_name)
load_dotenv(dotenv_path=config_file_path, override=True)

from utils.amlauth import AuthHelper
settings = AuthHelper.load_settings()
credential = AuthHelper.test_credential()

ml_client = MLClient(
    credential, settings.subscription_id, settings.resource_group, settings.workspace
)

In [3]:
# get the location of the current workspace
workspace_location = ml_client.workspaces.get(settings.workspace).location
print(f"Workspace location: {workspace_location}")

Workspace location: germanywestcentral


### Build context

`BuildContext(path=...)` uploads the entire folder to the workspace's ACR and runs
`docker build` against the `Dockerfile` inside it. The folder must contain:

- `Dockerfile` — the build recipe (`container/Dockerfile`)
- `requirements.txt` — pip deps that the Dockerfile `COPY`s in (`container/requirements.txt`)

Any other files in the folder become part of the build context.

In [4]:
# Verify the build context folder exists and lists the expected files.
build_context_path = os.path.join(".", "container")
dockerfile_path   = os.path.join(build_context_path, "Dockerfile")
requirements_path = os.path.join(build_context_path, "requirements.txt")

assert os.path.isdir(build_context_path), f"Missing folder: {build_context_path}"
assert os.path.isfile(dockerfile_path),   f"Missing file: {dockerfile_path}"
assert os.path.isfile(requirements_path), f"Missing file: {requirements_path}"

print(f"Build context : {os.path.abspath(build_context_path)}")
for f in sorted(os.listdir(build_context_path)):
    full = os.path.join(build_context_path, f)
    size = os.path.getsize(full) if os.path.isfile(full) else 0
    print(f"  📄 {f}  ({size} bytes)")

Build context : /Users/yingding/Code/VCS/ai/model-fine-tuning/02-compute-cluster/container
  📄 Dockerfile  (1872 bytes)
  📄 requirements.txt  (2825 bytes)


### Create / update the custom environment

Each `create_or_update` call with a new `version` produces a new immutable version
of the environment. Reference it later from a `command` job as
`environment="<env_name>@latest"` or `environment="<env_name>:<version>"`.

In [5]:
# Build a custom AML environment from container/Dockerfile + container/requirements.txt.
#
# WHY BuildContext (vs Environment(image=...)):
#   - image=...     : reuses an existing image as-is. Cannot add pip deps.
#   - BuildContext  : uploads the folder to the workspace ACR and runs `docker build`
#                     against the Dockerfile, producing a new image. This is the
#                     correct path when you need to layer custom pip packages
#                     (transformers, trl, peft, deepspeed, etc.) on top of an ACPT
#                     base image.
#
# The image build happens in the workspace's Azure Container Registry and may take
# several minutes the first time. Subsequent versions reuse cached layers when the
# Dockerfile / requirements.txt have not changed.
import datetime
from azure.ai.ml.entities import Environment, BuildContext

# ─── Knobs ────────────────────────────────────────────────────────────────
# Keep these in sync with the FROM line in container/Dockerfile.
# Used to derive the env name + the "base" tag for human-readable provenance.
CUDA_TAG     = "cu13"            # short tag for env name, e.g. cu121, cu126, cu13
TORCH_TAG    = "torch212"        # short tag, e.g. torch230, torch26, torch212
PY_TAG       = "py312"           # short tag, e.g. py310, py312
BASE_FAMILY  = "ngc-pytorch"     # source: nvcr.io / mcr.acpt / etc.

env_name        = f"sft-finetune-{CUDA_TAG}"
env_version     = datetime.datetime.now().strftime("%Y%m%d.%H%M")
env_description = (
    f"Custom {BASE_FAMILY} fine-tuning env ({CUDA_TAG}/{PY_TAG}/{TORCH_TAG}): "
    "transformers + trl + peft + deepspeed + bitsandbytes."
)

custom_env = Environment(
    name=env_name,
    version=env_version,
    description=env_description,
    build=BuildContext(
        path=build_context_path,    # folder containing Dockerfile + requirements.txt
        dockerfile_path="Dockerfile",
    ),
    tags={
        "app": "finetuning",
        "domain": "ml",
        "base": f"{BASE_FAMILY}-{CUDA_TAG}-{PY_TAG}-{TORCH_TAG}",
        "cuda": CUDA_TAG,
        "torch": TORCH_TAG,
        "python": PY_TAG,
    },
)

#### (Optional) Force a new version build for the same environment

If you make changes to the `Dockerfile` or `requirements.txt` but like use the same environment name, you can force a new version build by incrementing the `version` parameter in `create_or_update`. AML will detect the change in the build context and trigger a new ACR build.

In [9]:
FORCE_REBUILD = True  # set to True to force a new ACR build even if the same version already exists
# FORCE_REBUILD = False # set to False to skip build if the same version already exists (uses existing image in ACR if available, otherwise builds a new image)
if FORCE_REBUILD:
      print(f"Creating environment '{env_name}' version '{env_version}' ...")
      start_time = datetime.datetime.now()
      created_env = ml_client.environments.create_or_update(custom_env)
      end_time = datetime.datetime.now()

      print(f"✅ Submitted: {created_env.name}:{created_env.version}")
      print(f"   id            : {created_env.id}")
      print(f"   build context : {created_env.build.path if created_env.build else 'n/a'}")
      print(f"   elapsed       : {end_time - start_time}")
      print("\nNote: the actual image build runs asynchronously in ACR.")
      print("Track progress in Azure ML Studio → Environments → "
            f"{env_name}:{env_version} → Build log.")
else:
      print(f"Reusing environment '{env_name}' version '@latest'")

Creating environment 'sft-finetune-cu13' version '20260603.1701' ...


Uploading container (0.0 MBs): 100%|██████████| 4697/4697 [00:00<00:00, 39947.94it/s]




✅ Submitted: sft-finetune-cu13:20260603.1701
   id            : /subscriptions/6753a2ee-12b7-4fac-82fa-48824fb58abe/resourceGroups/rg-aml-yw-dos/providers/Microsoft.MachineLearningServices/workspaces/aml-ww-yw-dos/environments/sft-finetune-cu13/versions/20260603.1701
   build context : https://amlwwywdos1036066399.blob.core.windows.net/azureml-blobstore-fb4fd209-d5ec-483a-9266-a270f4c2fd57/LocalUpload/67e95a020c6123ab899bf028d7835dd933afee2734ea537d533c701841954cd2/container/
   elapsed       : 0:00:09.321910

Note: the actual image build runs asynchronously in ACR.
Track progress in Azure ML Studio → Environments → sft-finetune-cu13:20260603.1701 → Build log.


In [11]:
# Watch the env build via the AML Jobs API.
#
# `ml_client.environments.create_or_update(env_with_BuildContext)` doesn't
# only register the env — it also creates a hidden AML Job in the
# `prepare_image` experiment named `imgbldrun_<hash>` that runs the ACR
# build. That job is queryable via the regular jobs API.
#
# Strategy:
#   We just registered the env in the cell above, so the build job is the
#   MOST RECENT job in the `prepare_image` experiment. Grab the first match.
#   AML doesn't surface a deterministic link from env → build-job-name, and
#   the tag/display_name stamping varies across SDK versions, so "newest job
#   in the experiment" is the most reliable correlator.
#
# Caveats:
#   • azure-ai-ml 1.33 doesn't accept `experiment_name=` as a server-side
#     filter (TypeError from `requests.Session.request`). Filter client-side.
#   • If you run this cell well after env-create AND another build slipped
#     in (e.g. someone else in the workspace registered an env), the "newest"
#     may not be yours. Bump SCAN_LIMIT or use the tag-match fallback below.

import time

EXPERIMENT  = "prepare_image"
POLL_S      = 15
MAX_WAIT_S  = 1800   # 30 min ceiling
SCAN_LIMIT  = 50     # how many recent jobs to scan when looking for the build

print(f"Searching '{EXPERIMENT}' for the most recent build job ...")
build_job = None
seen = 0
for j in ml_client.jobs.list():     # newest first
    if seen >= SCAN_LIMIT:
        break
    seen += 1
    if (getattr(j, "experiment_name", "") or "") == EXPERIMENT:
        build_job = j
        break

if build_job is None:
    print(f"⚠️  No '{EXPERIMENT}' job found in the most recent {SCAN_LIMIT} jobs.")
    print("    The build may not have been enqueued yet — wait a few seconds and re-run.")
else:
    print(f"✅ Build job  : {build_job.name}")
    print(f"   display    : {build_job.display_name}")
    print(f"   created    : {getattr(build_job, 'creation_context', None) and build_job.creation_context.created_at}")
    print(f"   status     : {build_job.status}")
    print(f"   studio URL : {build_job.studio_url}")

    # Poll until terminal. (Use ml_client.jobs.stream(build_job.name) for
    # a live tail instead — that blocks and prints stdout as it appears.)
    terminal = {"Completed", "Failed", "Canceled", "NotResponding"}
    deadline = time.time() + MAX_WAIT_S
    last_status = None
    while time.time() < deadline:
        j = ml_client.jobs.get(build_job.name)
        if j.status != last_status:
            print(f"  status={j.status}")
            last_status = j.status
        if j.status in terminal:
            build_job = j
            break
        time.sleep(POLL_S)
    else:
        print("⚠️  Timed out — check Studio.")

    print(f"\nFinal status : {build_job.status}")

    # On failure, stream the logs so the Dockerfile error is visible
    # without leaving the notebook. (`stream` exits cleanly when the job
    # has already terminated — it just prints what it has.)
    if build_job.status != "Completed":
        print("\n──── build job logs ────")
        try:
            ml_client.jobs.stream(build_job.name)
        except Exception as e:
            print(f"⚠️  Could not stream logs: {e}")

        print("\n" + "═" * 70)
        print(f"❌ ENV BUILD FAILED   status={build_job.status}")
        print("═" * 70)
        print(f"Inspect in Studio: {build_job.studio_url}")
    else:
        print(f"\n✅ Env build succeeded.")
        print(f"Studio: {build_job.studio_url}")

Searching 'prepare_image' for the most recent build job ...
✅ Build job  : imgbldrun_a0ffd52
   display    : olive_fish_jhf478tj
   created    : 2026-06-03 15:02:35.279568+00:00
   status     : Queued
   studio URL : https://ml.azure.com/runs/imgbldrun_a0ffd52?wsid=/subscriptions/6753a2ee-12b7-4fac-82fa-48824fb58abe/resourcegroups/rg-aml-yw-dos/workspaces/aml-ww-yw-dos&tid=787eb5ff-f2ee-4965-8074-edbba3402c84
  status=Queued
  status=Running
  status=Finalizing
  status=Completed

Final status : Completed

✅ Env build succeeded.
Studio: https://ml.azure.com/runs/imgbldrun_a0ffd52?wsid=/subscriptions/6753a2ee-12b7-4fac-82fa-48824fb58abe/resourcegroups/rg-aml-yw-dos/workspaces/aml-ww-yw-dos&tid=787eb5ff-f2ee-4965-8074-edbba3402c84


### Verify the environment registration

After the ACR build succeeds, confirm the new env version is registered and
shows up via `environments.list / .get(label="latest")`. The next cell also
prints the image URI so you can sanity-check the ACR repository / tag.

In [8]:
# List all versions of the new env and show the latest.
#
# Note on `.image`:
#   For a `BuildContext` environment, the SDK's `Environment.image` attribute
#   stays `None` even after the ACR build succeeds — it's only populated when
#   the env was created via `Environment(image=...)`. The actual ACR image URI
#   for built envs lives in the workspace ACR under
#   `azureml/azureml_<hash>:latest` and is not surfaced by `environments.get`.
#   The authoritative "is it ready?" signal is the `prepare_image` build job
#   status, captured in `build_job` above.
versions = list(ml_client.environments.list(name=env_name))
print(f"Found {len(versions)} version(s) for '{env_name}':")
for e in versions:
    print(f"  - {e.name}:{e.version}")

latest_env = ml_client.environments.get(name=env_name, label="latest")
print(f"\nLatest  : {latest_env.name}:{latest_env.version}")
print(f"id      : {latest_env.id}")
print(f"tags    : {latest_env.tags}")

# Report build readiness from the build job (the real source of truth),
# falling back to a hint if we don't have one in scope.
build_status = locals().get("build_job", None)
if build_status is not None:
    ready = build_status.status == "Completed"
    icon  = "✅" if ready else "⏳" if build_status.status not in {"Failed", "Canceled", "NotResponding"} else "❌"
    print(f"build   : {icon} {build_status.status}  (job: {build_status.name})")
    if ready:
        print("        → image is built and pullable from the workspace ACR.")
else:
    print("build   : (status unknown — re-run the build-poll cell to refresh)")


Found 4 version(s) for 'sft-finetune-cu13':
  - sft-finetune-cu13:20260603.1027
  - sft-finetune-cu13:20260602.2319
  - sft-finetune-cu13:20260602.2051
  - sft-finetune-cu13:20260602.1939

Latest  : sft-finetune-cu13:20260603.1027
id      : /subscriptions/6753a2ee-12b7-4fac-82fa-48824fb58abe/resourceGroups/rg-aml-yw-dos/providers/Microsoft.MachineLearningServices/workspaces/aml-ww-yw-dos/environments/sft-finetune-cu13/versions/20260603.1027
tags    : {'app': 'finetuning', 'domain': 'ml', 'base': 'ngc-pytorch-cu13-py312-torch212', 'cuda': 'cu13', 'torch': 'torch212', 'python': 'py312'}
build   : ✅ Completed  (job: imgbldrun_f84c462)
        → image is built and pullable from the workspace ACR.


In [ ]:
# Optional: archive an old version (cannot be deleted, only archived).
# ml_client.environments.archive(name=env_name, version="<old_version>")

# Optional: restore an archived version.
# ml_client.environments.restore(name=env_name, version="<old_version>")

### Smoke-test the built environment

Once the ACR build cell (above) prints `✅ ACR build succeeded`, the env image
is in the workspace ACR and ready to use. The next cell submits a tiny
diagnostic job that:

1. Runs on `Cluster-A100-1GPU` against the just-built image.
2. Prints `nvidia-smi`, torch CUDA build info, bitsandbytes import result,
   and HF stack versions — so you can verify the runtime container actually
   has the GPU stack you expect.

If you submit this **before** the ACR build finishes, the job will sit in
`Preparing` waiting for the image — harmless, just slower feedback.

In [9]:
# Submit a tiny smoke-test job that uses the newly-created env.
#
# Auth model on a user-managed GPU cluster:
#   - `identity=UserIdentityConfiguration()` makes the JOB RUNTIME use your
#     Entra token (the cluster MI impersonates you for data-plane reads).
#   - HOWEVER, the AML v2 Execution control plane still calls `listKeys` on
#     the workspace storage account during orchestration. So shared-key MUST
#     remain enabled on the storage (see Step 2b + the exemption md cell).
#   - Fully identity-only jobs are only possible on serverless compute, not on
#     a user-managed cluster.
#
# DIAGNOSTIC MODE: this job dumps everything we need to debug the bnb /
# libcudart mismatch — no need to pull the ~25 GB ACR image locally. The
# script is split across an external file (./container_smoke.py) baked into
# the build context so we don't have to fight quoting hell with `python -c`.

import datetime
import textwrap
from pathlib import Path

from azure.ai.ml import command
from azure.ai.ml.entities import UserIdentityConfiguration

SMOKE_COMPUTE = "Cluster-A100-1GPU"
job_timestamp = datetime.datetime.now().strftime("%Y%m%d.%H%M%S")
smoke_display_name = f"env-smoke-{env_name}-{env_version}-{job_timestamp}"

# Write a small diagnostic script into the build context. AML's `command` with
# `code=None` doesn't have files to call, so we instead inline a script via
# heredoc inside bash — but escape pain has bitten us twice now. Cleaner: drop
# the python script into a folder, point `code=` at it, and call it by path.
smoke_dir = Path("./smoke")
smoke_dir.mkdir(exist_ok=True)
(smoke_dir / "smoke.py").write_text(textwrap.dedent("""
    import importlib
    import os
    import subprocess
    import sys
    import sysconfig

    def header(s):
        print("\\n" + "═" * 70)
        print(s)
        print("═" * 70)

    header("python")
    print(f"sys.executable     : {sys.executable}")
    print(f"sys.version        : {sys.version.split()[0]}")
    print(f"site-packages root : {sysconfig.get_paths()['purelib']}")

    header("env vars (CUDA-relevant)")
    for k in ("CUDA_HOME", "CUDA_PATH", "LD_LIBRARY_PATH", "BNB_CUDA_VERSION",
             "NVIDIA_VISIBLE_DEVICES", "NVIDIA_DRIVER_CAPABILITIES"):
        print(f"  {k:30s} = {os.environ.get(k, '(unset)')}")

    header("where is libcudart.so.* on disk?")
    subprocess.run(["bash", "-lc",
                    "find / -name 'libcudart.so*' 2>/dev/null | head -50"])

    header("ldconfig -p | grep cudart")
    subprocess.run(["bash", "-lc", "ldconfig -p | grep -i cudart || echo '(none)'"])

    header("nvidia-* pip packages")
    subprocess.run(["bash", "-lc", "pip list 2>/dev/null | grep -i ^nvidia || echo '(none)'"])

    header("bitsandbytes package files")
    subprocess.run(["bash", "-lc",
                    "ls -la /usr/local/lib/python3.12/dist-packages/bitsandbytes/ 2>/dev/null | head -40 "
                    "|| ls -la $(python -c 'import bitsandbytes, os; print(os.path.dirname(bitsandbytes.__file__))') 2>/dev/null "
                    "|| echo '(bnb dir not found)'"])

    header("torch CUDA build info")
    try:
        import torch
        print(f"torch.__version__         : {torch.__version__}")
        print(f"torch.version.cuda        : {torch.version.cuda}")
        print(f"torch.backends.cudnn.ver  : {torch.backends.cudnn.version()}")
        print(f"torch.cuda.is_available() : {torch.cuda.is_available()}")
        if torch.cuda.is_available():
            print(f"  device 0               : {torch.cuda.get_device_name(0)}")
            cap = torch.cuda.get_device_capability(0)
            print(f"  capability             : {cap}  (SM{cap[0]*10+cap[1]})")
    except Exception as e:
        print(f"torch import failed: {e}")

    header("bitsandbytes import attempt")
    try:
        import bitsandbytes as bnb
        print(f"✅ bnb {bnb.__version__} imported cleanly")
    except Exception as e:
        print(f"❌ {type(e).__name__}: {e}")
        # On failure, dlopen each candidate so we see the real linker error.
        import ctypes
        for cand in ("libcudart.so.12", "libcudart.so.13", "libcudart.so"):
            try:
                ctypes.CDLL(cand)
                print(f"   ✅ dlopen({cand}) OK")
            except OSError as oe:
                print(f"   ❌ dlopen({cand}): {oe}")

    header("nvidia-smi")
    subprocess.run(["bash", "-lc", "nvidia-smi || echo '(nvidia-smi not available)'"])

    header("other relevant package versions")
    for pkg in ("transformers", "accelerate", "datasets", "trl", "peft"):
        try:
            m = importlib.import_module(pkg)
            print(f"  {pkg:13s}: {getattr(m, '__version__', '(no __version__)')}")
        except Exception as e:
            print(f"  {pkg:13s}: MISSING ({e})")
"""))

smoke_job = command(
    code=str(smoke_dir),                 # upload ./smoke/ folder
    command="python smoke.py",
    environment=f"{env_name}@latest",
    compute=SMOKE_COMPUTE,
    identity=UserIdentityConfiguration(),
    display_name=smoke_display_name,
    experiment_name="env-smoke-tests",
    description=f"Diagnostic dump of {env_name}:{env_version} — libcudart / bnb / torch.",
)

print(f"Submitting smoke job for env '{env_name}@latest' on '{SMOKE_COMPUTE}' ...")
print(f"Display name : {smoke_display_name}")
print(f"Code dir     : {smoke_dir.resolve()}")
print("Runtime auth : UserIdentityConfiguration  |  Control plane: shared-key (per exemption)")
submitted = ml_client.jobs.create_or_update(smoke_job)
print(f"✅ Job submitted")
print(f"   name        : {submitted.name}")
print(f"   status      : {submitted.status}")
print(f"   studio URL  : {submitted.studio_url}")

Class AutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteConditionSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


Submitting smoke job for env 'sft-finetune-cu13@latest' on 'Cluster-A100-1GPU' ...
Display name : env-smoke-sft-finetune-cu13-20260603.1032-20260603.103657
Code dir     : /Users/yingding/Code/VCS/ai/model-fine-tuning/02-compute-cluster/smoke
Runtime auth : UserIdentityConfiguration  |  Control plane: shared-key (per exemption)


Class BaseAutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class IntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class ProtectionLevelSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseIntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


✅ Job submitted
   name        : ashy_turnip_5hwkg1tdrb
   status      : Starting
   studio URL  : https://ml.azure.com/runs/ashy_turnip_5hwkg1tdrb?wsid=/subscriptions/6753a2ee-12b7-4fac-82fa-48824fb58abe/resourcegroups/rg-aml-yw-dos/workspaces/aml-ww-yw-dos&tid=787eb5ff-f2ee-4965-8074-edbba3402c84


In [10]:
# Fetch the smoke-job result + logs.
#
# Two ways to get the output:
#   1. ml_client.jobs.stream(name)  — live tail (blocks until job ends). Best
#      when the job is still Running/Preparing and you want to watch.
#   2. ml_client.jobs.download(name, output_name="user_logs", download_path=...)
#      — downloads the captured logs for a completed job. Best after the fact.
#
# We do both:
#   - poll until the job hits a terminal state,
#   - then print the studio URL + status + tail of std_log.txt locally.

import time
from pathlib import Path

POLL_INTERVAL_S = 15
MAX_WAIT_S      = 1800  # 30 min ceiling

job_name = submitted.name  # set by the previous (smoke-submit) cell
print(f"Polling job '{job_name}' ...")

deadline = time.time() + MAX_WAIT_S
terminal = {"Completed", "Failed", "Canceled", "NotResponding"}
while time.time() < deadline:
    j = ml_client.jobs.get(job_name)
    print(f"  status={j.status}")
    if j.status in terminal:
        break
    time.sleep(POLL_INTERVAL_S)
else:
    print("⚠️  Timed out waiting for terminal state — check Studio.")

print(f"\nFinal status : {j.status}")
print(f"Studio URL   : {j.studio_url}")

# Download all logs to ./.job-logs/<job_name>/ (creates the dir).
log_root = Path(".job-logs") / job_name
log_root.mkdir(parents=True, exist_ok=True)
print(f"\nDownloading logs → {log_root.resolve()} ...")
try:
    ml_client.jobs.download(name=job_name, download_path=str(log_root), all=True)
    print("✅ Downloaded.")
except Exception as e:
    print(f"⚠️  Download failed: {e}")

# Print the user log tail. AML compute-cluster jobs put stdout/stderr at
# named-outputs/user_logs/std_log.txt (single-node) or std_log_process_<n>.txt
# (multi-node).
candidates = sorted(log_root.rglob("std_log*.txt"))
if not candidates:
    print(f"\n⚠️  No std_log*.txt under {log_root}. Listing what was downloaded:")
    for p in sorted(log_root.rglob("*")):
        if p.is_file():
            print(f"   {p.relative_to(log_root)}  ({p.stat().st_size} bytes)")
else:
    for log_path in candidates:
        text = log_path.read_text(errors="replace")
        tail = text.splitlines()[-200:]
        print(f"\n──── {log_path.relative_to(log_root)} (last {len(tail)} lines) ────")
        print("\n".join(tail))

Polling job 'ashy_turnip_5hwkg1tdrb' ...
  status=Queued
  status=Queued
  status=Queued
  status=Queued
  status=Queued
  status=Queued
  status=Queued
  status=Queued
  status=Queued
  status=Queued
  status=Queued
  status=Running
  status=Running
  status=Running
  status=Running
  status=Running
  status=Running
  status=Running
  status=Running
  status=Running
  status=Running
  status=Running
  status=Running
  status=Running
  status=Running
  status=Running
  status=Running
  status=Running
  status=Running
  status=Running
  status=Completed

Final status : Completed
Studio URL   : https://ml.azure.com/runs/ashy_turnip_5hwkg1tdrb?wsid=/subscriptions/6753a2ee-12b7-4fac-82fa-48824fb58abe/resourcegroups/rg-aml-yw-dos/workspaces/aml-ww-yw-dos&tid=787eb5ff-f2ee-4965-8074-edbba3402c84



✅ Downloaded.

──── artifacts/user_logs/std_log.txt (last 144 lines) ────
/usr/local/cuda-13.2/targets/x86_64-linux/lib/libcudart.so.13.2.75
/usr/local/cuda-13.2/targets/x86_64-linux/lib/libcudart.so.13
/usr/local/cuda-13.2/targets/x86_64-linux/lib/libcudart.so
	libcudart.so.13 (libc6,x86-64) => /usr/local/cuda/targets/x86_64-linux/lib/libcudart.so.13
	libcudart.so (libc6,x86-64) => /usr/local/cuda/targets/x86_64-linux/lib/libcudart.so
nvidia-cudnn-frontend              1.23.0
nvidia-cutlass-dsl                 4.4.2
nvidia-cutlass-dsl-libs-base       4.4.2
nvidia-dali-cuda130                2.1.0
nvidia-libnvcomp-cu13              5.1.0.21
nvidia-matmul-heuristics           0.1.0.27
nvidia-ml-py                       13.595.45
nvidia-modelopt                    0.43.0
nvidia-nvimgcodec-cu13             0.7.0.49
nvidia-nvjpeg                      13.1.0.48
nvidia-nvjpeg2k-cu13               0.10.0.49
nvidia-nvtiff-cu13                 0.7.0.79
nvidia-resiliency-ext              0.5.0
t